# RAG chatbot demonstration
This notebook demonstates the capability of a chat bot with RAG

In [1]:
import logging
import os

import certifi
import polars as pl
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from electricity_maps.config import get_settings

# Suppress HTTP call logs
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("chromadb").setLevel(logging.WARNING)
logging.getLogger("langchain").setLevel(logging.WARNING)

e:\IntelliJ\intelliWorkspace\ElectricityMaps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Setup Environment and Helper Functions
os.environ["AWS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()

get_settings.cache_clear()
settings = get_settings()
DATA_DIR = settings.data_dir

def get_sql_df(sql_query):
    # Use polars SQL context to query Delta Lake tables
    ctx = pl.SQLContext()

    # We'll automatically register tables from the 'gold' schema
    # by looking for gold.table_name in the query.
    gold_tables = ["daily_electricity_mix", "daily_imports", "daily_exports"]

    registered_any = False
    for table in gold_tables:
        path = f"{DATA_DIR}/gold/{table}"
        try:
            # scan_delta is lazy and efficient
            lf = pl.scan_delta(path, storage_options=settings.storage_options)
            # Register with 'gold.' prefix as the LLM uses that schema
            ctx.register(f"gold.{table}", lf)
            # Also register without prefix just in case
            ctx.register(table, lf)
            registered_any = True
        except Exception:
            # If the table doesn't exist yet, just skip it
            pass

    if not registered_any:
        print(f"Warning: No tables were successfully registered from {DATA_DIR}/gold/. Check your S3 credentials and data path.")

    # Execute the query directly. Polars SQL engine requires double quotes for table names with dots.
    # We automatically wrap gold.table_name with double quotes if they are missing.
    for table in gold_tables:
        pattern = f"gold.{table}"
        if pattern in sql_query and f'"{pattern}"' not in sql_query:
            sql_query = sql_query.replace(pattern, f'"{pattern}"')

    print(f"Executing SQL: {sql_query}")
    df = ctx.execute(sql_query, eager=True)
    return df.to_pandas()

def print_sql_df(sql_query):
    df = get_sql_df(sql_query)
    display(df)
    return df

gold_table = get_sql_df('SELECT count(*) as count FROM "gold.daily_electricity_mix"')
display(gold_table)

Executing SQL: SELECT count(*) as count FROM "gold.daily_electricity_mix"


,count
0,2


In [3]:
# 2. Data schema and LLM Initialization
gold_schemas = {
    "daily_electricity_mix": {
        "description": "Daily electricity mix data with percentages of different energy sources",
        "columns": ["process_ts", "zone", "zone_name", "date", "nuclear_pct", "biomass_pct", "wind_pct", "solar_pct", "hydro_pct", "gas_pct", "oil_pct", "coal_pct", "geothermal_pct", "unknown_pct", "total_production_mwh", "fossil_free_avg_pct", "renewable_avg_pct", "hours_covered", "year", "month", "day"]
    },
    "daily_imports": {
        "description": "Daily electricity imports data",
        "columns": ["process_ts", "zone", "zone_name", "source_zone", "source_zone_name", "date", "import_mwh", "hours_covered", "year", "month", "day"]
    },
    "daily_exports": {
        "description": "Daily electricity exports data",
        "columns": ["process_ts", "zone", "zone_name", "destination_zone", "destination_zone_name", "date", "export_mwh", "hours_covered", "year", "month", "day"]
    }
}

print(f"Model: {settings.llm_model}")
llm = ChatGoogleGenerativeAI(api_key=settings.llm_api_key, model=settings.llm_model)

schema_context = "\n".join([f"Table: gold.{table}\nDescription: {info['description']}\nColumns: {', '.join(info['columns'])}\n" for table, info in gold_schemas.items()])


Model: gemini-3.1-flash-lite-preview


In [4]:
# 3. Process docs folder and create embeddings in ChromaDB
docs_path = "../docs/Electricity_Maps_doc.pdf"
loader = PyPDFLoader(docs_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)

# Using all-MiniLM-L6-v2 embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory="./chroma_db")
retriever = vectorstore.as_retriever()


2026-04-27 23:15:11,905 - sentence_transformers.base.model - INFO - No device provided, using cpu
2026-04-27 23:15:12,222 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-27 23:15:12,502 - sentence_transformers.base.model - INFO - Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10120.25it/s]


In [5]:
# 4. Agent Setup: Router, Text-to-SQL, and RAG

# --- Router ---
router_template = """You are a query routing agent.
Given the following user query, decide if it should be routed to a structured database (SQL) or an unstructured document database (RAG).
Route to 'SQL' if the query asks for metrics, trends, percentages, data aggregation, imports/exports, or numerical electricity data.
Route to 'RAG' if the query asks for definitions, documentation, company information, methodology, FAQs, or explanations.

Return ONLY the string 'SQL' or 'RAG'.
Query: {query}
Route:"""
router_prompt = PromptTemplate.from_template(router_template)
router_chain = router_prompt | llm | StrOutputParser()

# --- Text-to-SQL Chain ---
sql_template = """You are a data analyst. Convert the natural language question into a valid SQL query for Polars SQL.
Only use the following schemas and tables.
Schema:
{schema}

IMPORTANT:
1. Always output ONLY the raw SQL query, with no markdown formatting or explanations. Do not include ```sql blocks.
    2. Always use double quotes for table names that include dots, like "gold.daily_electricity_mix".
3. If there's a reference to a latest process_ts, you can optionally filter by it if relevant.

Question: {question}
SQL Query:"""
sql_prompt = PromptTemplate.from_template(sql_template)
sql_chain = sql_prompt | llm | StrOutputParser()

# --- Document RAG Chain ---
rag_template = """Answer the question based only on the following context:
{context}

Question: {question}
Answer:"""
rag_prompt = ChatPromptTemplate.from_template(rag_template)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [6]:
# 5. Main Chatbot Interface
def ask_chatbot(question: str):
    print(f"User: {question}")

    # 1. Route the query
    route = router_chain.invoke({"query": question}).strip().upper()
    print(f"[Router decision: {route}]")

    # 2. Execute path
    if route == "SQL":
        # Generate SQL
        sql_query = sql_chain.invoke({"schema": schema_context, "question": question}).strip()
        # Clean up markdown formatting if the LLM adds it anyway
        sql_query = sql_query.replace('```sql', '').replace('```', '').strip()
        print(f"Generated SQL: {sql_query}")

        try:
            print("Executing query against Gold Layer...")
            print_sql_df(sql_query)
        except Exception as e:
            print(f"Error executing SQL: {e}")

    else:
        # Document RAG
        print("Querying Document Knowledge Base...")
        response = rag_chain.invoke(question)
        print(f"Chatbot: {response}")
    print("-" * 50)


In [7]:
# 6. Test the Chatbot
# Test 1: Ask for structured data (SQL Route)
ask_chatbot("What is the average renewable percentage for France?")



2026-04-27 23:15:19,110 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


User: What is the average renewable percentage for France?


2026-04-27 23:15:25,023 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


[Router decision: SQL]
Generated SQL: SELECT AVG(renewable_avg_pct) FROM "gold.daily_electricity_mix" WHERE zone_name = 'France'
Executing query against Gold Layer...
Executing SQL: SELECT AVG(renewable_avg_pct) FROM "gold.daily_electricity_mix" WHERE zone_name = 'France'


,renewable_avg_pct
0,25.661247


--------------------------------------------------


In [8]:
# Test 2: Ask for unstructured documentation (RAG Route)
ask_chatbot("What is Electricity Maps methodology?")

2026-04-27 23:15:30,505 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


User: What is Electricity Maps methodology?


2026-04-27 23:15:38,658 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


[Router decision: RAG]
Querying Document Knowledge Base...
Chatbot: Electricity Maps employs a triple-layered methodological choice consisting of an attributional, location-based, and consumption-based approach.
--------------------------------------------------


In [10]:
# Test 3: Complex SQL query
ask_chatbot("Show me the top 3 destinations for electricity exported from France")

2026-04-27 23:16:51,034 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


User: Show me the top 3 destinations for electricity exported from France


2026-04-27 23:16:59,137 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


[Router decision: SQL]


Generated SQL: SELECT destination_zone_name, SUM(export_mwh) AS total_exported FROM "gold.daily_exports" WHERE zone_name = 'France' GROUP BY destination_zone_name ORDER BY total_exported DESC LIMIT 3
Executing query against Gold Layer...
Executing SQL: SELECT destination_zone_name, SUM(export_mwh) AS total_exported FROM "gold.daily_exports" WHERE zone_name = 'France' GROUP BY destination_zone_name ORDER BY total_exported DESC LIMIT 3


,destination_zone_name,total_exported
0,Great Britain,89488.0
1,Italy North,39864.0
2,LU,5770.0


--------------------------------------------------
